# Regression Evaluator

Compares a **candidate model** against a **baseline** on the same prompt set.

Typical use cases:
- Comparing a fine-tuned model against the base model
- Checking whether a new vLLM version changes output behaviour
- Cross-environment verification (DV → QA → IN promotion)
- Comparing two model sizes (e.g. Qwen3-4B vs Qwen3-14B)

**Metrics computed per prompt:**
| Metric | Meaning |
|---|---|
| `Exact_match` | Outputs are character-for-character identical |
| `ROUGEL_drift` | Output similarity (1.0 = identical, 0.0 = completely different) |
| `Latency_delta` | Candidate latency minus baseline latency (seconds) |
| `Completion_tokens_delta` | Token count change (positive = candidate is more verbose) |
| `Candidate_refusal` | Candidate refused to answer where baseline did not |

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelRegressionEvaluator

## 2. Define baseline and candidate

Use `temperature=0.0` to make outputs deterministic — at higher temperatures the same model gives different outputs each run, making `Exact_match` meaningless.

In [ ]:
# Example: cross-environment comparison (same model, DV vs QA)
BASELINE_URL  = "https://millmchatqwen25-backend.dat-aip-millm.dv.mbcp.cloud/"
CANDIDATE_URL = "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/"

# Example: cross-size comparison (uncomment to use instead)
# BASELINE_URL  = "https://chatqwen3-4-backend.dat-aip-millm.qa.mbcp.cloud/"
# CANDIDATE_URL = "https://chatqwen3-14-backend.dat-aip-millm.qa.mbcp.cloud/"

## 3. Define the prompt set

Cover a variety of task types to get a representative comparison.

In [ ]:
PROMPTS = [
    # Factual / Q&A
    "What is the capital of Portugal?",
    "Explain what a transformer model is in two sentences.",
    # Task following
    "List three benefits of using a RAG pipeline.",
    "Translate to French: 'The bank is closed on Sundays.'",
    # Domain-specific
    "O que é um crédito habitação?",
    "Qual é a diferença entre um depósito a prazo e uma conta poupança?",
    # Edge cases
    "2 + 2 =",
    "Write a one-line Python function that reverses a string.",
]

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelRegressionEvaluator(
    baseline_url=BASELINE_URL,
    candidate_url=CANDIDATE_URL,
    prompts=PROMPTS,
    temperature=0.0,   # deterministic outputs for fair comparison
    max_tokens=512,
)

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate()

print(f"\n{'#':<3}  {'Exact':>5}  {'ROUGE-L':>7}  {'Lat Δ':>8}  {'Tok Δ':>6}  Prompt")
print("-" * 75)
for i, r in enumerate(results):
    exact = "✓" if r["Exact_match"] else "✗"
    drift = f"{r['ROUGEL_drift']:.3f}" if r["ROUGEL_drift"] is not None else "  N/A"
    lat_d = f"{r['Latency_delta']:+.2f}s" if r["Latency_delta"] is not None else "  N/A"
    tok_d = f"{r['Completion_tokens_delta']:+d}" if r["Completion_tokens_delta"] is not None else " N/A"
    print(f"{i+1:<3}  {exact:>5}  {drift:>7}  {lat_d:>8}  {tok_d:>6}  {r['Prompt'][:40]}")

## 6. Inspect a specific diff

In [ ]:
# Find the most drifted response
drifted = [
    r for r in results
    if r["ROUGEL_drift"] is not None and not r["Exact_match"]
]
if drifted:
    worst = min(drifted, key=lambda r: r["ROUGEL_drift"])
    print(f"Most drifted prompt (ROUGE-L drift = {worst['ROUGEL_drift']:.3f}):")
    print(f"\nPrompt:    {worst['Prompt']}")
    print(f"\nBaseline:  {worst['Baseline_output']}")
    print(f"\nCandidate: {worst['Candidate_output']}")
else:
    print("All outputs are identical — no drift detected.")

## 7. Extract aggregate metrics

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)

print(f"Baseline model:              {metrics['Baseline_model']}")
print(f"Candidate model:             {metrics['Candidate_model']}")
print(f"Prompts evaluated:           {metrics['Prompts_evaluated']}")
print()
print(f"Exact match rate:            {metrics['Exact_match_rate']:.1%}")
print(f"Avg ROUGE-L drift:           {metrics['Avg_ROUGEL_drift']:.4f}  (1.0 = identical)")
print()
print(f"Avg baseline latency:        {metrics['Avg_baseline_latency']}s")
print(f"Avg candidate latency:       {metrics['Avg_candidate_latency']}s")
print(f"Avg latency delta:           {metrics['Avg_latency_delta']:+.3f}s  ({metrics['Avg_latency_delta_pct']:+.1f}%)")
print()
print(f"Avg baseline compl. tokens:  {metrics['Avg_baseline_completion_tokens']}")
print(f"Avg candidate compl. tokens: {metrics['Avg_candidate_completion_tokens']}")
print(f"Avg token delta:             {metrics['Avg_completion_tokens_delta']}")
print()
print(f"Baseline refusal rate:       {metrics['Baseline_refusal_rate']:.1%}")
print(f"Candidate refusal rate:      {metrics['Candidate_refusal_rate']:.1%}")

## 8. JSON schema compliance regression test

If your use case requires structured output, pass a `json_schema` to verify the candidate still respects it.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string"},
        "confidence": {"type": "number"},
    },
    "required": ["category", "confidence"],
}

structured_prompts = [
    "Classify the following as 'positive', 'negative' or 'neutral' with a confidence score: 'I love this product!'",
    "Classify the following as 'positive', 'negative' or 'neutral' with a confidence score: 'The service was terrible.'",
]

ev_schema = ModelRegressionEvaluator(
    baseline_url=BASELINE_URL,
    candidate_url=CANDIDATE_URL,
    prompts=structured_prompts,
    json_schema=schema,
    temperature=0.0,
)

res_schema = ev_schema.evaluate()
m_schema = ev_schema.extract_threshold_metrics(res_schema)

print(f"Baseline JSON valid rate:  {m_schema['Baseline_json_valid_rate']:.1%}")
print(f"Candidate JSON valid rate: {m_schema['Candidate_json_valid_rate']:.1%}")

## 9. Log regression results to MLflow

The flat `metrics` dict maps directly to `mlflow.log_metrics`. Log both runs to the same experiment to track model evolution over time.

In [ ]:
import mlflow

mlflow.set_experiment("regression_evaluation")
with mlflow.start_run(run_name=f"{metrics['Baseline_model']}_vs_{metrics['Candidate_model']}"):
    loggable = {k: v for k, v in metrics.items() if isinstance(v, (int, float)) and v is not None}
    mlflow.log_metrics(loggable)
    mlflow.log_param("baseline_url", BASELINE_URL)
    mlflow.log_param("candidate_url", CANDIDATE_URL)
    mlflow.log_param("prompts_count", len(PROMPTS))
    print("Logged to MLflow:", loggable)